# IEEE-CIS Fraud Detection: Numerical-Feature Submission
## DSCI 598 Capstone, Week 4
### Ben Miller

Companion notebook: `bm-fraud-numeric-tuning` (where the model type and settings below were chosen).

# Part 2: Submit the tuned numerical-feature model
> After identifying the optimal model type and parameters, create a new notebook on Kaggle to submit predictions from this model.

### Imports, seeds and constants

Paste the three `BEST_` values printed at the end of the tuning notebook. The feature settings must match the tuning notebook so the model sees the same columns.

In [1]:
import os
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Kaggle mounts the competition files here (read-only)
DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'

# Column names used throughout
ID_COLUMN = 'TransactionID'
TARGET = 'isFraud'
TIME_COLUMN = 'TransactionDT'      # seconds from an undisclosed starting point, not a real timestamp
AMOUNT_COLUMN = 'TransactionAmt'

# From the last code cell of bm-fraud-numeric-tuning
BEST_MODEL_NAME = 'lightgbm'
BEST_PARAMS = {'subsample': 0.8, 'reg_lambda': 5.0, 'num_leaves': 255, 'min_child_samples': 50, 'learning_rate': 0.02, 'colsample_bytree': 0.3}
BEST_N_ESTIMATORS = 327   # LightGBM only: average number of trees early stopping chose; None for other models

# Numerical features (the decision above): measurement-like columns only
NUMERIC_FAMILIES = ['C', 'D', 'V', 'dist']   # counts, time deltas, Vesta features, distances
NUMERIC_IDENTITY_COLUMNS = [f'id_{number:02d}' for number in range(1, 12)]  # id_01-id_11 are measurements; id_12 on are categories
OTHER_NUMERIC_COLUMNS = [AMOUNT_COLUMN, 'has_identity']

# V-column reduction
CORRELATION_SAMPLE_SIZE = 100_000   # rows used to measure correlations (feature selection only, not tuning)
HIGH_CORRELATION = 0.95             # V columns more correlated than this with one already kept are dropped

# Model settings that aren't tuned
TREE_MISSING_VALUE = -999      # scikit-learn trees get missing values replaced by this; it sits below every real value
LOGREG_MAX_ITER = 2000
LIGHTGBM_MAX_ROUNDS = 3000     # upper limit; early stopping picks the real number
EARLY_STOPPING_ROUNDS = 100

SUBMISSION_PATH = 'submission.csv'   # Kaggle offers files in /kaggle/working (the default folder) for submission

### Data-loading functions

In [2]:
def downcast_numeric_columns(frame):
    """
    Shrink float64 columns to float32 and integer columns to the smallest type that fits.
    The raw tables need several GB as float64; float32 roughly halves that, and tree
    models don't need the extra precision.
    """
    smaller_types = {}
    for column in frame.columns:
        if pd.api.types.is_float_dtype(frame[column]):
            smaller_types[column] = 'float32'
        elif pd.api.types.is_integer_dtype(frame[column]):
            smallest_fit = pd.to_numeric(frame[column], downcast='integer')
            smaller_types[column] = smallest_fit.dtype

    # one astype call for every column; assigning hundreds of columns one at a time fragments the DataFrame
    return frame.astype(smaller_types)


def standardize_identity_columns(identity):
    """
    Rename test_identity's 'id-01' style columns to train's 'id_01' spelling.
    The competition files use a hyphen in test and an underscore in train, so without
    this every identity feature would silently be missing in test after the merge.
    """
    renamed_columns = {}
    for column in identity.columns:
        if column.startswith('id-'):
            renamed_columns[column] = column.replace('id-', 'id_')
    return identity.rename(columns=renamed_columns)


def load_table(file_name):
    """One competition CSV, with its numeric columns downcast."""
    frame = pd.read_csv(os.path.join(DATA_DIR, file_name))
    return downcast_numeric_columns(frame)


def merge_identity(transactions, identity):
    """
    Left-join identity onto transactions by TransactionID, and flag which rows had a match.
    Only some transactions have identity data, so a left join keeps every transaction and
    leaves the identity columns missing for the rest.
    """
    # mark identity rows before the merge; transactions without a match end up missing, then 0
    flagged_identity = identity.assign(has_identity=1)
    merged = transactions.merge(flagged_identity, on=ID_COLUMN, how='left')
    merged['has_identity'] = merged['has_identity'].fillna(0).astype('int8')
    return merged


def load_split(split_name):
    """The merged transaction + identity table for 'train' or 'test'."""
    transactions = load_table(f'{split_name}_transaction.csv')
    identity = standardize_identity_columns(load_table(f'{split_name}_identity.csv'))
    return merge_identity(transactions, identity)


def memory_mb(frame):
    """How much memory a DataFrame uses, in megabytes."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

### Load train and test

Train is sorted by time, as in the tuning notebook, so the V-column correlation sample picks exactly the same rows.

In [3]:
train = load_split('train')
train = train.sort_values(TIME_COLUMN).reset_index(drop=True)
test = load_split('test')   # test keeps its original order, which the submission file has to match
print(f'train: {train.shape[0]:,} rows x {train.shape[1]} columns')
print(f'test:  {test.shape[0]:,} rows x {test.shape[1]} columns')

train: 590,540 rows x 435 columns
test:  506,691 rows x 434 columns


### Feature functions

The same functions as the tuning notebook. The V-column correlations use the same seed and the same training rows, so the same columns are kept.

In [4]:
def column_family(column):
    """The feature family a column belongs to, from its name: 'V' for V1-V339, 'dist' for dist1-dist2, and so on."""
    if column.startswith('id_'):
        return 'id'
    return column.rstrip('0123456789')  # 'V258' -> 'V', 'dist1' -> 'dist'


def numeric_feature_columns(frame):
    """
    The measurement-like columns: amount, the identity flag, C, D, V, dist and id_01-id_11.
    Columns that hold category codes stored as numbers (card1-card6, addr1, addr2, id_12 on)
    are left out on purpose, along with the text columns.
    """
    features = []
    for column in frame.columns:
        if column_family(column) in NUMERIC_FAMILIES:
            features.append(column)
        elif column in NUMERIC_IDENTITY_COLUMNS or column in OTHER_NUMERIC_COLUMNS:
            features.append(column)
    return features


def select_uncorrelated_columns(frame, columns, threshold, sample_size):
    """
    Keep one column from each group of near-duplicates, scanning the columns in order.
    A column is dropped when its absolute correlation with a column already kept is above threshold.
    Correlations come from a random sample of rows: plenty to measure a correlation, and much faster than all rows.
    """
    sample = frame[columns].sample(n=min(sample_size, len(frame)), random_state=SEED)
    correlation = sample.corr().abs()
    kept_columns = []
    for column in columns:
        is_near_duplicate = False
        for kept_column in kept_columns:
            if correlation.loc[column, kept_column] > threshold:
                is_near_duplicate = True
                break
        if not is_near_duplicate:
            kept_columns.append(column)
    return kept_columns


def model_feature_columns(frame):
    """The numerical features with redundant V columns removed."""
    numeric_columns = numeric_feature_columns(frame)
    v_columns = []
    other_columns = []
    for column in numeric_columns:
        if column_family(column) == 'V':
            v_columns.append(column)
        else:
            other_columns.append(column)
    kept_v_columns = select_uncorrelated_columns(frame, v_columns, HIGH_CORRELATION, CORRELATION_SAMPLE_SIZE)
    return other_columns + kept_v_columns

### Build the numerical feature set

In [5]:
feature_columns = model_feature_columns(train)
print(f'{len(feature_columns)} features (should match the count printed in the tuning notebook)')

272 features (should match the count printed in the tuning notebook)


### Model builders

In [6]:
def build_logistic_regression(params):
    """Median imputation and scaling, since logistic regression handles neither missing values nor mixed scales."""
    return make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        LogisticRegression(max_iter=LOGREG_MAX_ITER, **params),   # ** passes each tuned setting as its own argument
    )


def build_decision_tree(params):
    """A single decision tree, with missing values replaced by a stand-in value."""
    return make_pipeline(
        SimpleImputer(strategy='constant', fill_value=TREE_MISSING_VALUE),
        DecisionTreeClassifier(random_state=SEED, **params),
    )


def build_random_forest(params):
    """A random forest, with missing values replaced by a stand-in value. n_jobs=-1 uses every CPU core."""
    return make_pipeline(
        SimpleImputer(strategy='constant', fill_value=TREE_MISSING_VALUE),
        RandomForestClassifier(random_state=SEED, n_jobs=-1, **params),
    )


def build_lightgbm(params):
    """LightGBM handles missing values itself. Its tree count is a ceiling that early stopping cuts short."""
    return lgb.LGBMClassifier(
        n_estimators=LIGHTGBM_MAX_ROUNDS,
        subsample_freq=1,      # subsample only takes effect when this is 1 or more
        metric='auc',          # early stopping watches AUC, not the default log loss
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
        **params,
    )


MODEL_BUILDERS = {
    'logistic_regression': build_logistic_regression,
    'decision_tree': build_decision_tree,
    'random_forest': build_random_forest,
    'lightgbm': build_lightgbm,
}

### Train on all training data and write the submission

The final model trains on all of train with the chosen settings. For LightGBM there's no validation slice left for early stopping, so it uses the average number of trees early stopping chose during tuning.

In [7]:
def build_final_model(model_name, params, n_estimators):
    """The chosen model type with its tuned settings. LightGBM gets a fixed tree count instead of early stopping."""
    model = MODEL_BUILDERS[model_name](params)
    if model_name == 'lightgbm':
        model.set_params(n_estimators=n_estimators)
    return model


final_model = build_final_model(BEST_MODEL_NAME, BEST_PARAMS, BEST_N_ESTIMATORS)
final_model.fit(train[feature_columns], train[TARGET])
test_fraud_probabilities = final_model.predict_proba(test[feature_columns])[:, 1]

submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], TARGET: test_fraud_probabilities})
submission.to_csv(SUBMISSION_PATH, index=False)

# The submission must have one row per test transaction, in the same order as Kaggle's sample file
sample_submission = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
same_ids = np.array_equal(submission[ID_COLUMN].to_numpy(), sample_submission[ID_COLUMN].to_numpy())
print(f'wrote {SUBMISSION_PATH}: {len(submission):,} rows; matches sample_submission IDs: {same_ids}')
print(submission.head().to_string(index=False))

wrote submission.csv: 506,691 rows; matches sample_submission IDs: True
 TransactionID  isFraud
       3663549 0.004732
       3663550 0.004617
       3663551 0.008359
       3663552 0.005673
       3663553 0.007500


### Kaggle score

- Model: **LightGBM** with numerical features only (272 columns), trained on all 590,540 training transactions with 327 trees.
- Public score **0.9115**, private score **0.8936**.
- Compared with the tuning notebook's last-fold AUC of **0.8960**, the private score is only 0.0024 lower. Validating on the most recent transactions gave a close estimate of how the model does on the later, hidden test data.